# `smooth_eval.py`

按功能分段；每段前有标题，方便查找和修改。

### Imports & constants

In [1]:
# smooth_eval.py
# 对已训练好的神经网络仓位做「事后降换手」处理（不需要重训），并在不偷看测试集的前提下评估：
#   1) 指数平滑（half-life 天）  2) 不交易区间（信号变化小于 theta 就不调仓）
#
# 为避免用测试结果挑参数，采用「前半段选参、后半段检验」：
#   - 在 TUNE_FOLDS 上、按事先声明的成本 C_TUNE_BPS 选出最优 (half-life, band)
#   - 只在 HOLD_FOLDS 上报告这一组参数的结果，并与「不处理」和 Long-Only 对比
# 需要：mlp_oos_predictions.csv / lstm_oos_predictions.csv（含 fold 列）

import os
import numpy as np
import pandas as pd

TRADING_DAYS = 252
TARGET_ANNUAL_VOL = 0.15
TARGET_DAILY_VOL = TARGET_ANNUAL_VOL / np.sqrt(TRADING_DAYS)

NN_FILES = {"MLP": "mlp_oos_predictions.csv", "LSTM": "lstm_oos_predictions.csv"}

TUNE_FOLDS = [1, 2]          # 用来选参数
HOLD_FOLDS = [3, 4]          # 只用来检验
C_TUNE_BPS = 2.0             # 选参数时假设的单边成本（事先声明，别根据结果回头改）
EVAL_COSTS = [0, 1, 2, 3, 5, 10]

HALFLIVES = [0, 3, 5, 10, 21, 42]     # 0 = 不平滑
BANDS = [0.0, 0.1, 0.2, 0.3]          # 0 = 不设区间

### `load_panel()`

In [2]:
def load_panel(path):
    p = pd.read_csv(path, parse_dates=["Date"]).sort_values(["Ticker", "Date"]).reset_index(drop=True)
    p["n_t"] = p.groupby("Date")["Ticker"].transform("count")
    p["code"] = p["Ticker"].astype("category").cat.codes
    return p

### `smooth_positions()`

In [3]:
def smooth_positions(p, halflife, band):
    sig = p["signal"]
    if halflife > 0:
        sig = sig.groupby(p["Ticker"]).transform(lambda s: s.ewm(halflife=halflife, adjust=False).mean())
    sig = sig.to_numpy(dtype=float)
    if band > 0:
        codes = p["code"].to_numpy()
        out = np.empty_like(sig)
        cur, prev = 0.0, -1
        for i in range(len(sig)):
            if codes[i] != prev:
                cur, prev = sig[i], codes[i]          # 新资产首日直接建仓
            elif abs(sig[i] - cur) >= band:
                cur = sig[i]
            out[i] = cur
        sig = out
    return pd.Series(sig, index=p.index)

### `daily_gross_and_turnover()`

In [4]:
def daily_gross_and_turnover(p, weight):
    gross = weight * p["fwd_ret_1d"] / p["n_t"]
    dw = weight.groupby(p["Ticker"]).diff().abs().fillna(0.0) / p["n_t"]
    return pd.DataFrame({"Date": p["Date"], "gross": gross, "turn": dw}).groupby("Date").sum()

### `rescale()`

In [5]:
def rescale(r, span=60):
    vol = r.ewm(span=span, adjust=False).std() * np.sqrt(TRADING_DAYS)
    return (r * (TARGET_ANNUAL_VOL / vol).shift(1)).dropna()

### `sharpe()`

In [6]:
def sharpe(r):
    return r.mean() / r.std() * np.sqrt(TRADING_DAYS)

### `net_sharpe()`

In [7]:
def net_sharpe(daily, c_bps, dates):
    net = daily["gross"] - (c_bps / 1e4) * daily["turn"]
    r = rescale(net)
    return sharpe(r[r.index.isin(dates)])

### `turnover()`

In [8]:
def turnover(daily, dates):
    return daily.loc[daily.index.isin(dates), "turn"].mean() * TRADING_DAYS

### `run_model()`

In [9]:
def run_model(name, p):
    fold_of = p.groupby("Date")["fold"].first()
    tune_dates = fold_of.index[fold_of.isin(TUNE_FOLDS)]
    hold_dates = fold_of.index[fold_of.isin(HOLD_FOLDS)]

    lev = TARGET_DAILY_VOL / p["vol_daily"]

    # ---- 1) 在 TUNE_FOLDS 上选参数
    grid = []
    for hl in HALFLIVES:
        for band in BANDS:
            d = daily_gross_and_turnover(p, smooth_positions(p, hl, band) * lev)
            grid.append({"halflife": hl, "band": band,
                         "tune_sharpe": net_sharpe(d, C_TUNE_BPS, tune_dates),
                         "tune_turnover": turnover(d, tune_dates)})
    grid = pd.DataFrame(grid)
    best = grid.sort_values("tune_sharpe", ascending=False).iloc[0]
    print(f"\n===== {name} =====")
    print(f"选参数区间: folds {TUNE_FOLDS}, 假设成本 {C_TUNE_BPS} bp")
    print(f"选中: half-life={int(best['halflife'])} 天, band={best['band']:.1f} "
          f"(选参区间净 Sharpe {best['tune_sharpe']:.3f}, 年化换手 {best['tune_turnover']:.1f})")

    # ---- 2) 只在 HOLD_FOLDS 上检验
    variants = {
        "不处理": (0, 0.0),
        f"平滑后(hl={int(best['halflife'])}, band={best['band']:.1f})": (int(best["halflife"]), float(best["band"])),
    }
    rows = {}
    for label, (hl, band) in variants.items():
        d = daily_gross_and_turnover(p, smooth_positions(p, hl, band) * lev)
        row = {"年化换手": turnover(d, hold_dates)}
        for c in EVAL_COSTS:
            row[f"{c}bp"] = net_sharpe(d, c, hold_dates)
        rows[f"{name} {label}"] = row

    d = daily_gross_and_turnover(p, pd.Series(1.0, index=p.index) * lev)       # Long-Only
    row = {"年化换手": turnover(d, hold_dates)}
    for c in EVAL_COSTS:
        row[f"{c}bp"] = net_sharpe(d, c, hold_dates)
    rows["Long-Only"] = row

    table = pd.DataFrame(rows).T
    print(f"\n检验区间: folds {HOLD_FOLDS}（缩放到 15% 波动率后的 Sharpe，列 = 单边成本）")
    print(table.round(3).to_string())
    grid.to_csv(f"{name.lower()}_smoothing_grid.csv", index=False)
    return table

### `main()`

In [10]:
def main():
    for name, path in NN_FILES.items():
        if not os.path.exists(path):
            print(f"[跳过] 找不到 {path}")
            continue
        run_model(name, load_panel(path))
    print("\n提示：选参数只用了 TUNE_FOLDS，检验区间没有参与选择；")
    print("      如果看完结果再回头改 C_TUNE_BPS 或网格，这个保护就失效了。")

### Entry point

In [11]:
if __name__ == "__main__":
    main()


===== MLP =====
选参数区间: folds [1, 2], 假设成本 2.0 bp
选中: half-life=0 天, band=0.3 (选参区间净 Sharpe 0.510, 年化换手 33.1)

检验区间: folds [3, 4]（缩放到 15% 波动率后的 Sharpe，列 = 单边成本）
                            年化换手    0bp    1bp    2bp    3bp    5bp   10bp
MLP 不处理                  110.776  0.539  0.221 -0.097 -0.415 -1.049 -2.622
MLP 平滑后(hl=0, band=0.3)   93.580  0.502  0.233 -0.036 -0.304 -0.840 -2.170
Long-Only                  4.708  0.586  0.580  0.574  0.568  0.555  0.524

===== LSTM =====
选参数区间: folds [1, 2], 假设成本 2.0 bp
选中: half-life=10 天, band=0.3 (选参区间净 Sharpe 0.649, 年化换手 0.9)

检验区间: folds [3, 4]（缩放到 15% 波动率后的 Sharpe，列 = 单边成本）
                             年化换手    0bp    1bp    2bp    3bp    5bp   10bp
LSTM 不处理                   53.957  0.713  0.502  0.292  0.081 -0.341 -1.391
LSTM 平滑后(hl=10, band=0.3)   1.650  0.694  0.686  0.679  0.671  0.655  0.617
Long-Only                   4.710  0.586  0.580  0.574  0.567  0.555  0.524

提示：选参数只用了 TUNE_FOLDS，检验区间没有参与选择；
      如果看完结果再回头改 C_TUNE_BPS 或网格，这个保护就失效